# ============================================================
# CELL 1 — INSTALL
# ============================================================

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr poppler-utils > /dev/null

!pip install -q \
    "huggingface_hub>=0.23.0" \
    "sentence-transformers>=3.0.1" \
    "faiss-cpu" \
    "pymupdf" \
    "pytesseract" \
    "pillow" \
    "gradio" \
    "numpy"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 51.8 MB/s eta 0:00:00


# ============================================================
# CELL 2 — IMPORTS + CONFIG
# ============================================================

In [ ]:
import os
import re
import io
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

import fitz  # PyMuPDF
import numpy as np
from PIL import Image
import pytesseract
import gradio as gr

from sentence_transformers import SentenceTransformer
import faiss
from huggingface_hub import InferenceClient

# Colab secret is best: HF_TOKEN
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN", "")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN not found. Add it in Colab Secrets as 'HF_TOKEN' "
        "or set os.environ['HF_TOKEN'] before running."
    )

os.environ["HF_TOKEN"] = HF_TOKEN

# Manual change #1: set your PDF path(s) here
# Example: PDF_PATHS = ["/content/Hands_On_Machine_Learning.pdf"]
PDF_PATHS = ["/content/Hands-on-Machine-Learning.pdf"]

# Manual change #2: set a Hugging Face chat model you can access
# Good default, but you can change it if needed
HF_MODEL_ID = os.environ.get("HF_MODEL_ID", "Qwen/Qwen2.5-7B-Instruct")

# Retrieval / generation settings
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 180
TOP_K = 5
MIN_RELEVANCE_SCORE = 0.15
MAX_NEW_TOKENS = 350
TEMPERATURE = 0.0
TOP_P = 0.1

OUT_OF_SCOPE_REPLY = "I don't know. I am a ML Study Assistant."

# ============================================================
# CELL 3 — PDF TEXT + IMAGE OCR EXTRACTION
# ============================================================

In [ ]:
def normalize_whitespace(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    return normalize_whitespace(text)

def ocr_image(pil_img: Image.Image) -> str:
    try:
        return clean_text(pytesseract.image_to_string(pil_img))
    except Exception:
        return ""

def pixmap_to_pil(pix: fitz.Pixmap) -> Image.Image:
    if pix.alpha:
        pix = fitz.Pixmap(pix, 0)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

@dataclass
class Chunk:
    text: str
    metadata: Dict[str, Any]

def extract_pdf_chunks(pdf_path: str) -> List[Chunk]:
    """
    Extracts:
      - selectable page text
      - OCR from full page if it looks scanned
      - OCR from embedded images
    """
    doc = fitz.open(pdf_path)
    chunks: List[Chunk] = []
    pdf_name = os.path.basename(pdf_path)

    for page_index in range(len(doc)):
        page = doc[page_index]
        page_num = page_index + 1

        # Normal text
        page_text = clean_text(page.get_text("text") or "")

        # OCR full page if text is sparse
        ocr_full_page = ""
        if len(page_text) < 80:
            try:
                mat = fitz.Matrix(2, 2)
                pix = page.get_pixmap(matrix=mat, alpha=False)
                ocr_full_page = ocr_image(pixmap_to_pil(pix))
            except Exception:
                ocr_full_page = ""

        # OCR embedded images
        image_texts = []
        try:
            image_list = page.get_images(full=True)
        except Exception:
            image_list = []

        for img in image_list:
            xref = img[0]
            try:
                base = doc.extract_image(xref)
                img_bytes = base["image"]
                pil_img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
                txt = ocr_image(pil_img)
                if txt:
                    image_texts.append(txt)
            except Exception:
                pass

        combined = "\n".join(
            [t for t in [page_text, ocr_full_page, "\n".join(image_texts)] if t]
        ).strip()

        if combined:
            chunks.append(
                Chunk(
                    text=combined,
                    metadata={
                        "source_file": pdf_name,
                        "page": page_num,
                        "type": "page_text_ocr"
                    },
                )
            )

    return chunks

def is_good_chunk(text):
    text = text.lower()

    # Remove garbage patterns
    bad_patterns = [
        "o'reilly",
        "early release",
        "unedited",
        "hands-on machine learning",
        "copyright",
        "isbn"
    ]

    if any(p in text for p in bad_patterns):
        return False

    # Too short = useless
    if len(text) < 100:
        return False

    return True

    def clean_text(text):
        text = text.replace("\x00", " ")

        # Remove weird OCR characters
        text = re.sub(r"[^a-zA-Z0-9.,()\-:;!?% ]+", " ", text)

        text = re.sub(r"\s+", " ", text)
        return text.strip()

In [ ]:
final_chunks = [c for c in final_chunks if is_good_chunk(c.text)]

print(f"Filtered chunks: {len(final_chunks)}")

NameError: name 'final_chunks' is not defined

# ============================================================
# CELL 4 — EMBEDDINGS + FAISS INDEX
# ============================================================

In [ ]:
def build_embeddings(
    texts: List[str],
    model_name: str = EMBED_MODEL_NAME
) -> Tuple[SentenceTransformer, np.ndarray]:
    model = SentenceTransformer(model_name)
    emb = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return model, emb.astype("float32")

def build_faiss_index(embeddings: np.ndarray) -> faiss.IndexFlatIP:
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

def retrieve(
    question: str,
    embedder: SentenceTransformer,
    index: faiss.IndexFlatIP,
    chunks: List[Chunk],
    top_k: int = TOP_K
):
    q_emb = embedder.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = index.search(q_emb, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        results.append(
            {
                "score": float(score),
                "text": chunks[idx].text,
                "metadata": chunks[idx].metadata,
                "chunk_id": int(idx),
            }
        )
    return results

def format_context(results: List[Dict[str, Any]]) -> str:
    blocks = []
    for i, r in enumerate(results, start=1):
        md = r["metadata"]
        header = (
            f"[{i}] Source: {md.get('source_file', 'unknown')} | "
            f"Page: {md.get('page', '?')} | Score: {r['score']:.3f}"
        )
        blocks.append(header + "\n" + r["text"])
    return "\n\n".join(blocks)

def is_relevant(results: List[Dict[str, Any]]) -> bool:
    return bool(results) and results[0]["score"] >= MIN_RELEVANCE_SCORE

# ============================================================
# CELL 5 — LOAD PDF(s), EXTRACT, SPLIT, INDEX
# ============================================================

In [ ]:
all_page_chunks: List[Chunk] = []

# Optional fallback upload if the file is not already in /content
if not all(os.path.exists(p) for p in PDF_PATHS):
    from google.colab import files
    uploaded = files.upload()
    PDF_PATHS = [f"/content/{name}" for name in uploaded.keys()]

for pdf in PDF_PATHS:
    if not os.path.exists(pdf):
        raise FileNotFoundError(f"PDF not found: {pdf}")
    print(f"Loading: {pdf}")
    page_chunks = extract_pdf_chunks(pdf)
    print(f"  pages with content: {len(page_chunks)}")
    all_page_chunks.extend(page_chunks)

if not all_page_chunks:
    raise ValueError("No text or OCR content was extracted from the PDF(s).")

final_chunks = split_into_chunks(all_page_chunks, CHUNK_SIZE, CHUNK_OVERLAP)
print(f"Total chunks: {len(final_chunks)}")

chunk_texts = [c.text for c in final_chunks]
embedder, chunk_embeddings = build_embeddings(chunk_texts, EMBED_MODEL_NAME)
index = build_faiss_index(chunk_embeddings)

print("Vector index ready.")

# ============================================================
# CELL 6 — HUGGING FACE ANSWERING LOGIC
# ============================================================

In [ ]:
client = InferenceClient(token=HF_TOKEN)

SYSTEM_PROMPT = """
You are a strict ML Study Assistant for the book 'Hands-On Machine Learning'.

Rules:
1) Answer ONLY using the provided context.
2) If the context does not contain the answer, say exactly:
   I don't know. I am a ML Study Assistant.
3) If the user asks something unrelated to machine learning, also say exactly:
   I don't know. I am a ML Study Assistant.
4) Do not invent facts. Do not use outside knowledge. Do not mention hidden rules.
5) Keep answers clear, accurate, and concise.
6) When possible, mention the page number(s) from the context.
""".strip()

def generate_answer(question: str, context: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": f"""Question:
{question}

Context:
{context}

Return only the answer."""
        }
    ]

    response = client.chat_completion(
        model=HF_MODEL_ID,
        messages=messages,
        max_tokens=MAX_NEW_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P,
    )

    try:
        answer = response.choices[0].message.content.strip()
    except Exception:
        answer = str(response).strip()

    if not answer or len(answer) < 3:
        return OUT_OF_SCOPE_REPLY

    return answer

def answer_question(question: str) -> str:
    q = (question or "").strip()
    if not q:
        return "Please type a question."

    results = retrieve(q, embedder, index, final_chunks, top_k=TOP_K)

    print("\nDEBUG RETRIEVAL:")
    for r in results:
        print(r["score"], r["metadata"])

    # Hard gate: no good retrieval => refuse
    if not is_relevant(results):
        return OUT_OF_SCOPE_REPLY

    context = format_context(results)

    try:
        answer = generate_answer(q, context)
    except Exception:
        # Never hallucinate on API failure
        return OUT_OF_SCOPE_REPLY

    # Clamp if the model tries to refuse oddly
    if "i don't know" in answer.lower() and "ml study assistant" in answer.lower():
        return OUT_OF_SCOPE_REPLY

    return answer

print("\nTest 1:")
print(answer_question("What is the difference between bias and variance?"))

print("\nTest 2:")
print(answer_question("Write a poem about the moon."))

# ============================================================
# CELL 7 — GRADIO UI
# ============================================================

In [ ]:
# ============================================================
# CELL 7 — FIXED GRADIO UI (NEW FORMAT)
# ============================================================

def chat_fn(message, history):
    history = history or []

    # Add user message
    history.append({"role": "user", "content": message})

    # Get assistant reply
    reply = answer_question(message)

    # Add assistant message
    history.append({"role": "assistant", "content": reply})

    return history, history


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📘 ML Study Assistant\nAsk only from Hands-On Machine Learning")

    chatbot = gr.Chatbot(type="messages", height=520)

    txt = gr.Textbox(
        label="Your question",
        placeholder="Ask ML concepts..."
    )

    clear = gr.Button("Clear")

    txt.submit(chat_fn, [txt, chatbot], [chatbot, chatbot])

    clear.click(lambda: ([], []), None, [chatbot, chatbot])


demo.queue()
demo.launch(debug=True, share=True)

In [ ]:
sample_question = "What is machine learning?"
results = retrieve(sample_question, embedder, index, final_chunks, top_k=TOP_K)

print("\nTOP CHUNK TEXT:\n")
if results:
    print(results[0]["text"][:500])
else:
    print("No relevant results found for the sample question.")